# DeepSCF: a periodic polymer
In this tutorial, you will predict the electron density of a periodic C/H polymer with the pretrained **(R+S)×3** model and use that density in a one-step non-SCF calculation. A conventional SCF calculation provides the reference density, total energy, and band structure.

Use a **CPU runtime** and run the cells in order. The first code cell downloads DeepSCF and all tutorial inputs; no manual upload or model training is needed.

**Workflow:** setup → inspect the structure and inputs → reference SCF → prepare grid data → predict the density → non-SCF evaluation.

*The calculation workflow was tested in Colab on 26 October 2026 with Python 3.13.16, PyTorch 2.11.0 (CPU), and sisl 0.20.0.*

## 1. Clone the repositories
Download the fixed DeepSCF snapshot and this tutorial into `/content`. DeepSCF provides the model and modified SIESTA source; `tutorial/deepscf/01/src` contains the polymer inputs and Python helpers.

The next cell checks the snapshot and required files. Existing clone directories are reused.

In [ ]:
%%bash
set -e
cd /content
if [ ! -d DeepSCF ]; then
  git clone https://github.com/yhkimlab/DeepSCF.git
  git -C DeepSCF checkout --detach fe3c3c18052b51101e438dad2ffa06f94e77b5bb
fi

if [ ! -d tutorial ]; then
  git clone --depth 1 --branch main https://github.com/YHKlab-RGLee/tutorial.git tutorial
fi


In [ ]:
%%bash
set -e
cd /content/DeepSCF
# Check the tutorial snapshot.
[ "$(git rev-parse HEAD)" = fe3c3c18052b51101e438dad2ffa06f94e77b5bb ]
test -s requirements.txt
test -s src/siesta-4.1-b4/Obj/gfortran-system.make
git log -1 --format='%h %cs %s'
test -s /content/tutorial/deepscf/01/src/inputs/common.fdf
test -s /content/tutorial/deepscf/01/src/predict_polymer.py
echo "DeepSCF and tutorial inputs ready"


## 2. Install dependencies
Use Colab's Python environment. PyTorch runs the pretrained network, h5py reads the grid dataset, and sisl reads SIESTA structures and output files. Matplotlib displays the structure, density, and bands inside the notebook.

Install the packages, then import them and check their versions.

In [ ]:
%pip install -q torch h5py hydra-core==1.3.2 omegaconf==2.3.0 "sisl[viz]" matplotlib

In [ ]:
%matplotlib inline
import sys, subprocess
import torch, h5py, numpy, hydra, omegaconf
import sisl, sisl.viz

print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| sisl:', sisl.__version__)

## 3. Build and launch SIESTA
Compile the modified SIESTA bundled with DeepSCF. This version writes the additional grids needed for preprocessing and reads the predicted density for the non-SCF calculation. Compilation may take a few minutes; the full log is saved to `/content/siesta-build.log`.

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq build-essential binutils gfortran libblas-dev liblapack-dev
cd /content/DeepSCF/src/siesta-4.1-b4/Obj
sh ../Src/obj_setup.sh
cp gfortran-system.make arch.make
# Save the build log.
make -j2 > /content/siesta-build.log 2>&1 || { tail -80 /content/siesta-build.log; exit 1; }
test -x siesta
echo "SIESTA build OK"


**Check the executable.** Launch SIESTA without an input file and inspect the version and compiler information. The missing-input error and nonzero exit code are expected here; this cell only checks that the executable starts.

In [ ]:
import subprocess, tempfile

with tempfile.TemporaryDirectory() as workdir:
    result = subprocess.run(
        ['/content/DeepSCF/src/siesta-4.1-b4/Obj/siesta'],
        stdin=subprocess.DEVNULL, capture_output=True, text=True,
        cwd=workdir, timeout=15,
    )
print(result.stdout)
print(result.stderr)
print('Exit code:', result.returncode, '(no input supplied)')


## 4. Reference calculation and preprocessing
### 4.1 Prepare the run folders
Copy the supplied inputs into separate SCF and non-SCF folders. Each run uses a new working directory, printed below, so previous output files are not reused.

To repeat the calculation, start again from this cell and run the subsequent cells.

In [ ]:
from pathlib import Path
import sys, shutil, tempfile

ROOT = Path('/content/DeepSCF')
SRC = Path('/content/tutorial/deepscf/01/src')
PYTHON = sys.executable
SIESTA = ROOT / 'src/siesta-4.1-b4/Obj/siesta'
(ROOT / 'results').mkdir(exist_ok=True)
WORK = Path(tempfile.mkdtemp(prefix='polymer-', dir=ROOT / 'results'))
SCF, NSCF = WORK / 'scf/polymer', WORK / 'non_scf'
for folder, mode in [(SCF, 'scf'), (NSCF, 'non_scf')]:
    folder.mkdir(parents=True)
    for name in ['STRUCT.fdf', 'common.fdf', 'C.psf', 'H.psf']:
        shutil.copy(SRC / 'inputs' / name, folder / name)
    shutil.copy(SRC / 'inputs' / mode / 'RUN.fdf', folder / 'RUN.fdf')
print('Working directory:', WORK)

### 4.2 Inspect the polymer structure
Read the supplied `STRUCT.fdf` with sisl. It contains **24 atoms (C₈H₁₆)** in a **20 × 20 × 10 Å** cell. The polymer extends periodically along **z**; the transverse directions contain vacuum.

The figure is an **x–z projection**, so atoms separated along y may overlap. Inspect the chain and the cell boundaries before running the calculation.

In [ ]:
geometry = sisl.get_sile(str(SCF / 'STRUCT.fdf')).read_geometry()
print('Number of atoms:', len(geometry))
print('Cell vectors (Angstrom):\n', geometry.cell)
geometry.plot(axes='xz', show_cell='box', backend='matplotlib').show()

### 4.3 Inspect the FDF inputs
SIESTA reads `RUN.fdf`, which includes `common.fdf`; the latter includes `STRUCT.fdf`. The files below define the structure, shared numerical settings, and SCF control.

| Input setting | Purpose in this example |
|---|---|
| `PAO.BasisSize DZP`, `XC.authors PBE` | Use a double-zeta polarized basis and PBE. |
| `kgrid_Monkhorst_Pack` | Sample the periodic chain with a 1 × 1 × 8 k-point mesh. |
| `MeshSizes 80 80 40` | Fix the grid at 0.25 Å spacing; explicit mesh sizes override the requested cutoff. |
| `MD.NumCGsteps 0` | Keep the supplied atomic positions fixed. |
| `SaveRho`, `SaveIonicCharge` | Write density grids used in preprocessing. |
| `MaxSCFIterations 300`, `SCF.MustConverge T` | Require the reference SCF calculation to converge. |

`BandLines` specifies Γ → X, where **X = (0, 0, ½)** in reciprocal-lattice coordinates along the polymer axis. The bands are folded into the supplied 10 Å repeat cell.

In [ ]:
for name in ['STRUCT.fdf', 'common.fdf', 'RUN.fdf']:
    print(f'--- {name} ---')
    print((SCF / name).read_text())

### 4.4 Run the reference SCF calculation
Run SIESTA with the inspected inputs. The helper checks successful completion and SCF convergence. Its detailed output is stored in `stdout.txt` and `stderr.txt` in the SCF folder.

The converged density `Molecular.RHO` and bands `Molecular.bands` will be used as references. The modified SIESTA also writes the grids consumed by DeepSCF preprocessing.

In [ ]:
subprocess.run([PYTHON, str(SRC / 'run_siesta.py'), str(SCF),
                '--siesta', str(SIESTA)], check=True)

### 4.5 Inspect the reference density
Compare the initial atomic density with the converged SCF density. The plots show the density integrated along y, in e/Å², rather than a single plane through the structure.

The reported relative error uses the **full 3D grid**:
\[
\epsilon_\rho = 100\,\frac{\sum_i |\rho_i-\rho_i^{\mathrm{SCF}}|}{\sum_i \rho_i^{\mathrm{SCF}}}.
\]
The uniform grid volume cancels in this ratio. The integrated SCF charge should be close to **48 valence electrons** for C₈H₁₆.

In [ ]:
sys.path.insert(0, str(SRC))
from analysis import plot_density, plot_bands, compare_energy

plot_density(SCF / 'Molecular.RHO', atomic=SCF / 'rhoatom');

### 4.6 Inspect the reference bands
Plot the bands near the gap along Γ–X. The energy zero is the **reference SCF valence-band maximum** on this path.

The printed gap is obtained from the sampled Γ–X eigenvalues. This is a path-based comparison, not a separate search over the full Brillouin zone.

In [ ]:
plot_bands(SCF);

### 4.7 Convert the grids into model inputs
Use the original tutorial scripts to convert the SIESTA grids and assemble `dataset/polymer.h5`. Apply the fixed scaling parameters supplied for the **(R+S)×3** model; no scaling parameters are fitted to this polymer.

The HDF5 file contains three input channels in `feature` and the SCF reference in `target`. Only `feature` enters the network; `target` is retained for evaluation. Check the array shapes printed below.

In [ ]:
subprocess.run([PYTHON, str(ROOT / 'tutorial/1.preprocess/1.convert.py'),
                str(WORK / 'scf'), str(WORK / 'raw')], check=True)
subprocess.run([PYTHON, str(ROOT / 'tutorial/1.preprocess/3.collect.py'),
                str(WORK / 'raw'), str(ROOT / 'datasets/scale/scale_r_plus_s_x3.txt'),
                str(WORK / 'dataset')], check=True)

with h5py.File(WORK / 'dataset/polymer.h5', 'r') as dataset:
    print('feature:', dataset['feature'].shape)
    print('target: ', dataset['target'].shape)

## 5. Predict the electron density
### 5.1 Download the pretrained model
Download the pretrained archive and keep only `pretrained/r_plus_s_x3/model.pt`. The download is about 486 MB and is skipped if the weights are already present.

This tutorial uses inference only: neither the training dataset nor a training run is required.

In [ ]:
%%bash
set -euo pipefail
cd /content/DeepSCF
if [ ! -s pretrained/r_plus_s_x3/model.pt ]; then
  curl -fL --retry 3 https://github.com/yhkimlab/DeepSCF/releases/download/DeepSCF_v1/pretrained-models-v1.tar.gz \
    | tar --no-same-owner -xz pretrained/r_plus_s_x3/model.pt
fi
test -s pretrained/r_plus_s_x3/model.pt


### 5.2 Run DeepSCF
Load the **(R+S)×3** model and predict the density on CPU. The helper writes `predict/polymer.RHO` using the same cell and grid as the reference calculation and checks the exported file.

Keep this predicted density for the next step; it will be passed to SIESTA as `In.RHO`.

In [ ]:
subprocess.run([PYTHON, str(SRC / 'predict_polymer.py'), str(ROOT), str(WORK)], check=True)
PREDICTED = WORK / 'predict/polymer.RHO'

### 5.3 Compare the predicted and SCF densities
Compare the atomic, predicted, and reference densities on the same grid. The final panel shows the projected difference **DeepSCF − SCF**.

Use the full-grid relative errors to quantify the improvement over the atomic starting density. The difference plot helps locate the regions where the prediction differs from the reference.

In [ ]:
plot_density(SCF / 'Molecular.RHO', atomic=SCF / 'rhoatom', prediction=PREDICTED);

## 6. Evaluate the predicted density with SIESTA
### 6.1 Prepare and inspect the non-SCF input
Copy the prediction to `In.RHO` in the non-SCF folder. The geometry, pseudopotentials, basis, grid, and k points are the same as in the reference calculation.

Inspect the non-SCF `RUN.fdf` below:
- `SCF.Read.Charge T` reads the supplied density.
- `MaxSCFIterations 1` limits the calculation to one iteration.
- `SCF.Mix.First F` disables first-step mixing.
- `SCF.MustConverge F` allows completion without SCF convergence.

The shared input also sets `DM.UseSaveDM F`; no converged density-matrix restart is copied into this folder.

In [ ]:
shutil.copy(PREDICTED, NSCF / 'In.RHO')
print('Density input:', NSCF / 'In.RHO')
print('--- non-SCF RUN.fdf ---')
print((NSCF / 'RUN.fdf').read_text())

### 6.2 Run the one-step calculation
Evaluate the electronic structure using the predicted density. The helper checks that SIESTA completes **exactly one SCF iteration**.

An `SCF_NOT_CONV` message is expected for this deliberately non-self-consistent step. The calculation is not continued to the reference SCF solution.

In [ ]:
subprocess.run([PYTHON, str(SRC / 'run_siesta.py'), str(NSCF),
                '--siesta', str(SIESTA), '--non-scf'], check=True)

### 6.3 Compare total energies and bands
Compare the total energy with the reference SCF result; the absolute difference is also reported in **meV per atom**. Then overlay the Γ–X bands using the same **SCF valence-band maximum** as the energy zero for both calculations.

Inspect the occupied and unoccupied bands as well as the printed path gaps. A small density error alone does not determine the accuracy of every electronic-structure observable.

In [ ]:
compare_energy(SCF, NSCF)
plot_bands(SCF, NSCF);

### Scope and output files
The fixed 0.25 Å grid follows the pretrained example; basis, mesh, and k-point convergence are not established here. The reference SCF is included for comparison, so the full workflow is not a speedup benchmark.

Logs: `stdout.txt` and `stderr.txt` in each run folder. Build log: `/content/siesta-build.log`.

[DeepSCF tutorial](https://github.com/yhkimlab/DeepSCF/blob/fe3c3c18052b51101e438dad2ffa06f94e77b5bb/tutorial/README.md) · [Polymer structure](https://github.com/yhkimlab/DeepSCF/blob/fe3c3c18052b51101e438dad2ffa06f94e77b5bb/tutorial/1.preprocess/structures/examples/polymer.fdf) · [sisl bands](https://sisl.readthedocs.io/en/latest/api/io/generated/sisl.io.siesta.bandsSileSiesta.html)
